<a href="https://colab.research.google.com/github/raz0208/Retrieval-Augmented-Generation-RAG-From-Scratch/blob/main/Retrieval_Augmented_Generation_(RAG)_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation(RAG) From Scratch
#### A traditional and basic RAG implementation

In this notebook, we will build a Retrieval-Augmented Generation(RAG) pipeline from scratch without using any popular libraries such as Langchain or Llamaindex.

RAG is a technique that retrieves related documents to the user's question, combines them with LLM-base prompt, and sends them to LLMs like GPT to produce more factually accurate generation.

## Lets Split RAG Pipeline into 5 parts:

1. Data loading
2. Chunking and Embedding
3. Vector Store
4. Retrieval & Prompt Preparation
5. Answer Generation

## Step 1: Install Dependences and Import Required Libraries

In [1]:
!pip install transformers scikit-learn docx2txt datasets nltk lancedb openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.8/314.8 kB 22.2 MB/s eta 0:00:00


## Step 2: Set OPENAI API KEY as env variable

In [2]:
# Setp OpenAI API Key
import os

os.environ["OPENAI_API_KEY"] = "sk-xxx"

In [3]:
# Load and read data pdf/txt

# !wget link
# !wget https://raw.githubusercontent.com/lancedb/vectordb-recipes/main/tutorials/RAG-from-Scratch/lease.txt
!wget https://www.gutenberg.org/cache/epub/80/pg80.txt

# Open and read the file - change the name of file related to the download link
with open("pg80.txt", "r") as file:
    text_data = file.read()

--2026-04-28 21:03:12--  https://www.gutenberg.org/cache/epub/80/pg80.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635356 (620K) [text/plain]
Saving to: ‘pg80.txt’

pg80.txt            100%[===================>] 620.46K  3.92MB/s    in 0.2s    

2026-04-28 21:03:20 (3.92 MB/s) - ‘pg80.txt’ saved [635356/635356]



In [4]:
# text_data
print(text_data[:500])

﻿The Project Gutenberg eBook of The Online World
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

**


## Data Chunking and Embedding

In [11]:
import nltk
import re
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

# Recursive Function To Split Texts and Creat Chunks
def recursive_text_splitter(text, max_chunk_length=1000, overlap=100):
    """
    Helper function for chunking text recursively
    """
    # Initialize result
    result = []

    current_chunk_count = 0
    separator = ["\n", " "]
    _splits = re.split(f"({separator})", text)
    splits = [_splits[i] + _splits[i + 1] for i in range(1, len(_splits), 2)]

    for i in range(len(splits)):
        if current_chunk_count != 0:
            chunk = "".join(
                splits[
                    current_chunk_count
                    - overlap : current_chunk_count
                    + max_chunk_length
                ]
            )
        else:
            chunk = "".join(splits[0:max_chunk_length])

        if len(chunk) > 0:
            result.append("".join(chunk))
        current_chunk_count += max_chunk_length

    return result

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [13]:
# Split the text using the recursive character text splitter
chunks = recursive_text_splitter(text_data, max_chunk_length=100, overlap=10)
print("Number of Chunks: ", len(chunks))
print("Number of Token Per Each Chunk:", len(chunks[0].split()))

Number of Chunks:  1298
Number of Token Per Each Chunk: 92
